In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [3]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

/Users/E2074/analytics/customer/appography/cleanup_activity


In [9]:
start_date = '20240715' 
end_date = '20240724'

In [10]:
#regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r'"(.*?)"', text)]

In [11]:
query = f"""

    select 
        id,
        apps_list
    from 
        (
        select
            id, yyyymmdd,
            json_extract(data, '$.appographyData') as apps_list
            -- row_number() over(partition by id order by updated_epoch desc) updated_seq
        from 
            raw_internal.kafka_info_customer_installed_apps_v1
        where 
            yyyymmdd BETWEEN '{start_date}'
            AND '{end_date}'
        )
    -- where
       -- updated_seq = 1
"""

df_data_raw = pd.read_sql(query, connection)

In [12]:
df_data_raw.head(5)

,id,apps_list
0,6614248da2e3de5a4166ccac,"[""Duolingo"","" Messenger"","" Google Drive"","" Goo..."
1,62c9a660551cdb02f8c0d2ee,"[""Zomato"","" BigBasket"","" Wynk Music"","" Domino'..."
2,6362bac785b399b62658418c,"[""Grofers delivery app"","" Zomato"","" Domino's P..."
3,627d8decc75b905bb908d3a5,"[""Facebook"","" Flipkart"","" Google Drive"","" Goog..."
4,623ac521b9a21e697dcbde95,"[""Airtel Thanks"","" Amazon Prime Video"","" Clock..."


In [13]:
df_data = df_data_raw.copy()

In [14]:
df_data['app_list_set'] = df_data['apps_list'].apply(lambda x: extract_and_trim(x))
df_data = df_data[['id', 'app_list_set']]

In [15]:
df_data.head(5)

,id,app_list_set
0,6614248da2e3de5a4166ccac,"[Duolingo, Messenger, Google Drive, Google Hom..."
1,62c9a660551cdb02f8c0d2ee,"[Zomato, BigBasket, Wynk Music, Domino's Pizza..."
2,6362bac785b399b62658418c,"[Grofers delivery app, Zomato, Domino's Pizza,..."
3,627d8decc75b905bb908d3a5,"[Facebook, Flipkart, Google Drive, Google Maps..."
4,623ac521b9a21e697dcbde95,"[Airtel Thanks, Amazon Prime Video, Clock, Ang..."


In [16]:
## Explode list into individual rows

df_data_explode = df_data.explode('app_list_set')
df_data_explode['app_list_set'] = df_data_explode['app_list_set'].str.lower()

df_data_explode.head(4)

,id,app_list_set
0,6614248da2e3de5a4166ccac,duolingo
0,6614248da2e3de5a4166ccac,messenger
0,6614248da2e3de5a4166ccac,google drive
0,6614248da2e3de5a4166ccac,google home


In [17]:
total_customers = df_data_explode.id.nunique()
total_customers

10311348

In [18]:
df_temp = df_data_explode\
            .groupby(['app_list_set'])\
            .agg(customers = ('id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,app_list_set,customers,%
0,google maps,10307688,99.96
1,youtube,10270401,99.60
2,gmail,10269261,99.59
3,whatsapp,10021891,97.19
4,google photos,9724178,94.31
5,instagram,8266859,80.17
6,google drive,7665706,74.34
7,google pay,7515895,72.89
8,facebook,7216195,69.98
9,truecaller,6911825,67.03


In [19]:
query_cat = f"""

SELECT 
    lower(app_name) app_name, 
    lower(category) category 
FROM 
    experiments_internal.customer_app_category_mapping
"""

df_data_cat = pd.read_sql(query_cat, connection)

In [20]:
df_data_cat.head(5)

,app_name,category
0,ignou e-content,educational
1,aha,ott
2,grofers delivery app,delivery_grocery
3,qr code reader,tools
4,nishtha,office


In [21]:
df_merge = pd.merge(df_temp,df_data_cat,how='left', left_on='app_list_set', right_on='app_name')

In [22]:
df_merge[['category', 'app_list_set', 'customers', '%']]

,category,app_list_set,customers,%
0,tools,google maps,10307688,99.96
1,tools,youtube,10270401,99.60
2,tools,gmail,10269261,99.59
3,social,whatsapp,10021891,97.19
4,tools,google photos,9724178,94.31
5,social,instagram,8266859,80.17
6,tools,google drive,7665706,74.34
7,finance_transactions,google pay,7515895,72.89
8,social,facebook,7216195,69.98
9,tools,truecaller,6911825,67.03


In [23]:
df_temp.to_clipboard(index=False)